**Legacy notebook.** Self-contained analysis code that predates the `src/mrvf` library and has not been ported to it. Kept for provenance and because it still produces figures in `results/`. Paths were updated to the `results/` layout; the next cell sets the working directory to the repository root, so run it from anywhere.

For the maintained pipeline see `notebooks/01_train_triple_regime.ipynb` and `notebooks/02_evaluate_rmse_vs_snr.ipynb`.

In [ ]:
import os
from pathlib import Path
# run from the repository root so ./results/... and ../subsamples resolve
_root = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src" / "mrvf").is_dir())
os.chdir(_root)

# Statistical Comparison of RMSE: DL Noise-Aware vs DL Noise-Free vs DM

This notebook performs pairwise statistical tests comparing per-sample absolute errors
between three reconstruction methods across four SNR levels.

**Methods compared:**
- DL Noise-Aware (Triple A+B+C, unified multi-SNR training)
- DL Noise-Free (Triple A+B+C, noise-free training)
- DM (normalized inner-product dictionary matching)

**Statistical test:** Two-sided Wilcoxon signed-rank test on paired per-sample
absolute errors (same test samples used for all methods). Bonferroni correction
applied across 4 parameters × 4 SNR levels = 16 comparisons per method pair.

**Effect size:** Cohen's d = (mean_error_A - mean_error_B) / pooled_std

**Requires** (adjust paths in CONFIG as needed):
- Noise-free dictionary: `QuasiRand_t2_200.mat`
- Noisy SNR dictionaries: `QuasiRand_t2_snr{SNR}.mat`
- Parameter file: `QuasiRand_par_t2_200.mat`
- NF model checkpoint: `triple_regime_nf_results_v1/models/triple_nf_best.pt`
- Noisy model checkpoint: `triple_regime_results_v1/models/triple_best.pt`
- Echo times: `echotimes.mat`

In [ ]:
# ── Cell 1: Imports ──────────────────────────────────────────────────────────
import os, json
import numpy as np
import scipy.io as sio
import h5py
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# ── Cell 2: CONFIG ───────────────────────────────────────────────────────────
CONFIG = {
    'param_path'          : '../subsamples/subsamples_v3/QuasiRand_par_t2_200.mat',
    'noisefree_sig_path'  : '../subsamples/subsamples_v3/QuasiRand_t2_200.mat',
    'dict_base_path'      : '../subsamples/subsamples_v3',
    'echotimes_path'      : '../echotimes.mat',
    'nf_model_ckpt'       : './results/triple_regime_nf_results_v1/models/triple_nf_best.pt',
    'noisy_model_ckpt'    : './results/triple_regime_results_v1/models/triple_regime_best.pt',
    'output_dir'          : './results/statistical_comparison',
    'dict_key'            : 'Dico40_save',
    'param_key'           : 'par_save',
    'param_mins'  : np.array([0.0,   0.0025,  1.0e-6,  0.050]),
    'param_maxs'  : np.array([1.0,   0.15,   25.0e-6,  0.200]),
    'param_names' : ['SO2', 'CBV', 'R', 'T2'],
    'n_fid'   : 14,
    'n_rephas': 16,
    'n_postse': 10,
    'R2starA_min':  2.0,  'R2starA_max': 55.0,
    'R2starB_min': -30.0, 'R2starB_max': 22.0,
    'R2starC_min':  2.0,  'R2starC_max': 55.0,
    'test_frac'  : 0.15,
    'val_frac'   : 0.15,
    'snr_levels' : [20, 50, 100, 150],
    # Number of test samples to use per SNR (None = all)
    'max_test_samples': 50000,
    'dm_chunk_size'   : 256,
}
SE_ECHO = CONFIG['n_fid'] + CONFIG['n_rephas']
PARAM_NAMES = ['SO\u2082', 'CBV', 'R', 'T2']
PARAM_UNITS = ['(%)', '(%)', '(\u03bcm)', '(ms)']
PARAM_SCALE = [100, 100, 1e6, 1000]
PKEYS = CONFIG['param_names']
N_COMPARISONS = len(PKEYS) * len(CONFIG['snr_levels'])  # for Bonferroni

os.makedirs(CONFIG['output_dir'], exist_ok=True)
print(f'Bonferroni correction across {N_COMPARISONS} tests per method pair')

In [ ]:
# ── Cell 3: Utility functions ─────────────────────────────────────────────────
def load_mat(path, key):
    try:
        mat = sio.loadmat(path)
        if key in mat: return np.array(mat[key], dtype=np.float32)
        cands = [k for k in mat if not k.startswith('_')]
        return np.array(mat[cands[0]], dtype=np.float32)
    except NotImplementedError:
        with h5py.File(path, 'r') as f:
            data = f[key][()] if key in f else f[next(k for k in f if not k.startswith('#'))][()]
            if data.ndim >= 2: data = data.T
            return np.array(data, dtype=np.float32)

def euclidean_norm(data):
    data = np.abs(data).astype(np.float32)
    return data / np.maximum(np.linalg.norm(data, axis=1, keepdims=True), 1e-12)

def params_scale(p, mins, maxs):
    return ((p - mins) / (maxs - mins)).astype(np.float32)

def params_inverse(p, mins, maxs):
    return (p * (maxs - mins) + mins).astype(np.float32)

def filter_param_range(signals, params, mins, maxs):
    mask = np.ones(len(params), dtype=bool)
    for i in range(min(params.shape[1], len(mins))):
        mask &= (params[:, i] >= mins[i]) & (params[:, i] <= maxs[i])
    return signals[mask], params[mask]

def clean_data(signals, params):
    valid = np.all(np.isfinite(signals), axis=1) & np.all(np.isfinite(params), axis=1)
    return signals[valid], params[valid]

def add_rician_noise(signals, snr):
    s0 = np.mean(np.abs(signals[:, 0]))
    sigma = s0 / snr
    nr = np.random.normal(0, sigma, signals.shape).astype(np.float32)
    ni = np.random.normal(0, sigma, signals.shape).astype(np.float32)
    return np.sqrt((signals + nr)**2 + ni**2).astype(np.float32)

def compute_triple_regime_features(signals, cfg):
    et_mat = sio.loadmat(cfg['echotimes_path'])
    et = et_mat['Echotimes'].flatten() / 1000.0
    tA = et[:cfg['n_fid']]
    tB = et[cfg['n_fid']:SE_ECHO]
    tC_rel = et[SE_ECHO:] - et[SE_ECHO - 1]
    def ols_rate(t, s):
        log_s = np.log(np.maximum(s, 1e-12))
        A = np.column_stack([t, np.ones_like(t)])
        slope = np.linalg.lstsq(A, log_s.T, rcond=None)[0][0]
        return -slope
    R2A = ols_rate(tA, np.abs(signals[:, :cfg['n_fid']]))
    R2B = ols_rate(tB, np.abs(signals[:, cfg['n_fid']:SE_ECHO]))
    R2C = ols_rate(tC_rel, np.abs(signals[:, SE_ECHO:]))
    return R2A.astype(np.float32), R2B.astype(np.float32), R2C.astype(np.float32)

def scale_triple_features(R2A, R2B, R2C, cfg):
    fA = np.clip((R2A - cfg['R2starA_min']) / (cfg['R2starA_max'] - cfg['R2starA_min']), 0, 1)
    fB = np.clip((R2B - cfg['R2starB_min']) / (cfg['R2starB_max'] - cfg['R2starB_min']), 0, 1)
    fC = np.clip((R2C - cfg['R2starC_min']) / (cfg['R2starC_max'] - cfg['R2starC_min']), 0, 1)
    return fA, fB, fC

def build_43dim_input(signals, cfg):
    sig_norm = euclidean_norm(signals)
    R2A, R2B, R2C = compute_triple_regime_features(signals, cfg)
    fA, fB, fC = scale_triple_features(R2A, R2B, R2C, cfg)
    valid = np.all(np.isfinite(sig_norm), axis=1) & np.isfinite(fA) & np.isfinite(fB) & np.isfinite(fC)
    X = np.column_stack([sig_norm, fA[:, None], fB[:, None], fC[:, None]]).astype(np.float32)
    return X, valid

def batched_predict(model, x_np, batch_size=8192):
    model.eval(); preds = []
    with torch.no_grad():
        for i in range(0, len(x_np), batch_size):
            xb = torch.tensor(x_np[i:i+batch_size], dtype=torch.float32).to(device)
            preds.append(model(xb).cpu().numpy())
    return np.concatenate(preds, axis=0)

# def dictionary_match(dict_norm, dict_params, test_norm, chunk_size=512):
#     best_idx = np.zeros(len(test_norm), dtype=np.int32)
#     for start in range(0, len(test_norm), chunk_size):
#         end = min(start + chunk_size, len(test_norm))
#         corr = test_norm[start:end] @ dict_norm.T
#         best_idx[start:end] = corr.argmax(axis=1)
#     return dict_params[best_idx]
def dictionary_match(dict_norm, dict_params, test_norm, chunk_size=512):
    """GPU-accelerated inner-product dictionary matching."""
    best_idx = np.zeros(len(test_norm), dtype=np.int32)

    # Move full dictionary to GPU once
    dict_t = torch.tensor(dict_norm, dtype=torch.float32).to(device)   # (N_dict, 40)

    with torch.no_grad():
        for start in range(0, len(test_norm), chunk_size):
            end   = min(start + chunk_size, len(test_norm))
            chunk = torch.tensor(
                test_norm[start:end], dtype=torch.float32
            ).to(device)                                                  # (chunk, 40)
            corr  = torch.mm(chunk, dict_t.T)                            # (chunk, N_dict)
            best_idx[start:end] = corr.argmax(dim=1).cpu().numpy()

    # Free GPU memory
    del dict_t
    torch.cuda.empty_cache()

    return dict_params[best_idx]

# def dictionary_match(dict_norm, dict_params, test_norm,
#                      vox_chunk=256, dict_chunk=100_000):
#     """GPU DM chunked on both voxel and dictionary dimensions to fit 6GB VRAM."""
#     best_idx   = np.zeros(len(test_norm), dtype=np.int32)
#     best_score = np.full(len(test_norm), -np.inf, dtype=np.float32)

#     with torch.no_grad():
#         for d_start in range(0, len(dict_norm), dict_chunk):
#             d_end  = min(d_start + dict_chunk, len(dict_norm))
#             dict_t = torch.tensor(
#                 dict_norm[d_start:d_end], dtype=torch.float32
#             ).to(device)                                         # (dict_chunk, 40)

#             for v_start in range(0, len(test_norm), vox_chunk):
#                 v_end  = min(v_start + vox_chunk, len(test_norm))
#                 vox_t  = torch.tensor(
#                     test_norm[v_start:v_end], dtype=torch.float32
#                 ).to(device)                                     # (vox_chunk, 40)

#                 corr   = torch.mm(vox_t, dict_t.T)              # (vox_chunk, dict_chunk)
#                 scores, indices = corr.max(dim=1)
#                 scores  = scores.cpu().numpy()
#                 indices = indices.cpu().numpy() + d_start

#                 better  = scores > best_score[v_start:v_end]
#                 best_score[v_start:v_end][better] = scores[better]
#                 best_idx[v_start:v_end][better]   = indices[better]

#             del dict_t
#             torch.cuda.empty_cache()

#     return dict_params[best_idx]

print('Utilities ready')

In [ ]:
class FiLMLayer(nn.Module):
    def __init__(self, feature_dim, n_features=3, hidden=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, hidden),
            nn.ReLU(),
            nn.Linear(hidden, feature_dim * 2)
        )
    def forward(self, x, regime):
        out = self.net(regime)
        gamma, beta = out.chunk(2, dim=-1)
        return x * (1 + gamma) + beta

class TripleRegimeModel(nn.Module):
    def __init__(self, n_outputs=4, dropout=0.05):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, 32, 7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(dropout),
            nn.Conv1d(32, 64, 5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(dropout),
            nn.Conv1d(64, 128, 3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Conv1d(128, 256, 3, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(256, 256, 3, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(),
        )
        self.fc1    = nn.Linear(1280, 512)
        self.bn1    = nn.BatchNorm1d(512)
        self.film1  = FiLMLayer(512)
        self.fc2    = nn.Linear(512, 256)
        self.bn2    = nn.BatchNorm1d(256)
        self.film2  = FiLMLayer(256)
        self.fc3    = nn.Linear(256, 128)
        self.bn3    = nn.BatchNorm1d(128)
        self.film3  = FiLMLayer(128)
        self.fc_out = nn.Linear(128, n_outputs)

    def forward(self, x):
        sig    = x[:, :40].unsqueeze(1)
        regime = x[:, 40:]
        h = self.conv(sig).flatten(1)
        h = self.film1(torch.relu(self.bn1(self.fc1(h))), regime)
        h = self.film2(torch.relu(self.bn2(self.fc2(h))), regime)
        h = self.film3(torch.relu(self.bn3(self.fc3(h))), regime)
        return torch.sigmoid(self.fc_out(h))

print('TripleRegimeModel defined')

In [ ]:
# Diagnostic — print actual checkpoint keys
ckpt = torch.load(CONFIG['nf_model_ckpt'], map_location='cpu')
for k, v in ckpt.items():
    print(f"{k:<50} {str(tuple(v.shape))}")

In [ ]:
# ── Cell 5: Load models ───────────────────────────────────────────────────────
model_nf = TripleRegimeModel(n_outputs=4, dropout=0.05).to(device)
model_nf.load_state_dict(torch.load(CONFIG['nf_model_ckpt'], map_location=device))
model_nf.eval()
print(f'NF model loaded from {CONFIG["nf_model_ckpt"]}')

model_noisy = TripleRegimeModel(n_outputs=4, dropout=0.05).to(device)
model_noisy.load_state_dict(torch.load(CONFIG['noisy_model_ckpt'], map_location=device))
model_noisy.eval()
print(f'Noisy model loaded from {CONFIG["noisy_model_ckpt"]}')

In [ ]:
# ── Cell 6: Load noise-free dictionary and build shared test set indices ───────
print('Loading noise-free dictionary...')
sig_nf  = load_mat(CONFIG['noisefree_sig_path'], CONFIG['dict_key'])
par_raw = load_mat(CONFIG['param_path'], CONFIG['param_key'])[:, :4]

sig_nf, par_raw = filter_param_range(sig_nf, par_raw, CONFIG['param_mins'], CONFIG['param_maxs'])
sig_nf, par_raw = clean_data(sig_nf, par_raw)
print(f'Dictionary: {len(sig_nf):,} entries')

# Reproduce the exact same train/val/test split as the training notebook
n = len(sig_nf)
idx_all = np.arange(n)
idx_tv, idx_test = train_test_split(idx_all, test_size=CONFIG['test_frac'], random_state=42)

# Subsample test set for speed if needed
rng = np.random.default_rng(0)
max_n = CONFIG['max_test_samples']
if max_n and len(idx_test) > max_n:
    idx_test = rng.choice(idx_test, max_n, replace=False)

sig_test  = sig_nf[idx_test]       # noise-free test signals
par_test  = par_raw[idx_test]      # ground truth parameters

# Scale ground truth to physical units
par_test_phys = par_test * np.array([100, 100, 1e6, 1000], dtype=np.float32)
print(f'Test set: {len(sig_test):,} samples')

In [ ]:
# ── Cell 7: Build DM dictionary (noise-free, normalized) ─────────────────────
print('Building DM dictionary...')
dict_norm = euclidean_norm(sig_nf)

# Scale DM params to physical units
par_raw_phys = par_raw * np.array([100, 100, 1e6, 1000], dtype=np.float32)
print(f'DM dictionary: {len(dict_norm):,} entries')

In [ ]:
# ── Cell 8: Compute per-sample absolute errors at each SNR ─────────────────────
# For each SNR: inject noise into test signals, run all 3 methods, record |error|

all_errors = {snr: {'DL_noisy': {}, 'DL_NF': {}, 'DM': {}} for snr in CONFIG['snr_levels']}

for snr in CONFIG['snr_levels']:
    print(f'\n── SNR = {snr} ──────────────────────────────')

    # Inject noise
    np.random.seed(snr)
    sig_noisy = add_rician_noise(sig_test.copy(), snr)

    # Build 43-dim input for both DL models
    X_noisy, valid = build_43dim_input(sig_noisy, CONFIG)
    X_noisy  = X_noisy[valid]
    par_eval = par_test_phys[valid]
    sig_eval = sig_noisy[valid]
    print(f'  Valid samples after feature extraction: {valid.sum():,}')

    # ── DL Noisy ──
    pred_noisy_scaled = batched_predict(model_noisy, X_noisy)
    pred_noisy_phys   = params_inverse(
        pred_noisy_scaled,
        CONFIG['param_mins'][:4],
        CONFIG['param_maxs'][:4]
    ) * np.array([100, 100, 1e6, 1000], dtype=np.float32)

    # ── DL NF ──
    pred_nf_scaled = batched_predict(model_nf, X_noisy)
    pred_nf_phys   = params_inverse(
        pred_nf_scaled,
        CONFIG['param_mins'][:4],
        CONFIG['param_maxs'][:4]
    ) * np.array([100, 100, 1e6, 1000], dtype=np.float32)

    # ── DM ──
    test_norm = euclidean_norm(sig_eval)
    # pred_dm_phys = dictionary_match(dict_norm, par_raw_phys, test_norm)
    pred_dm_phys = dictionary_match(dict_norm, par_raw_phys, test_norm,
                                 chunk_size=CONFIG['dm_chunk_size'])

    # Store per-sample absolute errors per parameter
    for j, pkey in enumerate(PKEYS):
        true_j = par_eval[:, j]
        all_errors[snr]['DL_noisy'][pkey] = np.abs(pred_noisy_phys[:, j] - true_j)
        all_errors[snr]['DL_NF'][pkey]    = np.abs(pred_nf_phys[:, j]    - true_j)
        all_errors[snr]['DM'][pkey]       = np.abs(pred_dm_phys[:, j]    - true_j)

    # Print RMSE summary for sanity check
    print(f'  {"Param":<6} {"DL_Noisy":>10} {"DL_NF":>10} {"DM":>10}')
    for j, (pkey, unit) in enumerate(zip(PKEYS, PARAM_UNITS)):
        r_noisy = float(np.sqrt(np.mean(all_errors[snr]['DL_noisy'][pkey]**2)))
        r_nf    = float(np.sqrt(np.mean(all_errors[snr]['DL_NF'][pkey]**2)))
        r_dm    = float(np.sqrt(np.mean(all_errors[snr]['DM'][pkey]**2)))
        print(f'  {pkey:<6} {r_noisy:>8.3f}{unit}  {r_nf:>8.3f}{unit}  {r_dm:>8.3f}{unit}')

print('\nDone.')

In [ ]:
# ── Cell 9: Wilcoxon signed-rank tests with Bonferroni correction ─────────────
# Two comparisons of interest:
#   (A) DL_noisy vs DM
#   (B) DL_noisy vs DL_NF
# Bonferroni threshold: alpha / (n_params * n_snr_levels)

alpha = 0.05
n_tests = len(PKEYS) * len(CONFIG['snr_levels'])
alpha_corrected = alpha / n_tests
print(f'Bonferroni-corrected alpha: {alpha_corrected:.5f}  ({n_tests} tests per pair)')

def cohens_d(a, b):
    diff = a - b
    return float(np.mean(diff)) / float(np.std(diff) + 1e-12)

rows = []
for comparison, (mA, mB) in [('DL_noisy vs DM', ('DL_noisy', 'DM')),
                               ('DL_noisy vs DL_NF', ('DL_noisy', 'DL_NF'))]:
    for snr in CONFIG['snr_levels']:
        for j, (pkey, unit) in enumerate(zip(PKEYS, PARAM_UNITS)):
            eA = all_errors[snr][mA][pkey]
            eB = all_errors[snr][mB][pkey]

            # Wilcoxon signed-rank test on paired per-sample differences
            stat, p_raw = stats.wilcoxon(eA, eB, alternative='two-sided')
            p_corr = min(p_raw * n_tests, 1.0)   # Bonferroni
            sig = '***' if p_corr < 0.001 else '**' if p_corr < 0.01 else '*' if p_corr < 0.05 else 'ns'
            d = cohens_d(eA, eB)
            rmse_A = float(np.sqrt(np.mean(eA**2)))
            rmse_B = float(np.sqrt(np.mean(eB**2)))

            rows.append({
                'Comparison': comparison,
                'SNR': snr,
                'Parameter': f'{pkey} {unit}',
                'RMSE_A': round(rmse_A, 3),
                'RMSE_B': round(rmse_B, 3),
                'p_raw': f'{p_raw:.2e}',
                'p_Bonferroni': f'{p_corr:.2e}',
                'Significance': sig,
                "Cohen's d": round(d, 3),
            })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

df.to_csv(os.path.join(CONFIG['output_dir'], 'statistical_comparison.csv'), index=False)
print(f'\nSaved to {CONFIG["output_dir"]}/statistical_comparison.csv')

In [ ]:
# ── Cell 10: Summary table per comparison ────────────────────────────────────
for comp in df['Comparison'].unique():
    print(f'\n{"="*70}')
    print(f'  {comp}')
    print(f'{"="*70}')
    sub = df[df['Comparison'] == comp]
    print(sub[['SNR','Parameter','RMSE_A','RMSE_B','p_Bonferroni','Significance',"Cohen's d"]].to_string(index=False))

In [ ]:
# ── Cell 11: Significance heatmap ────────────────────────────────────────────
import matplotlib.colors as mcolors

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
comparisons = df['Comparison'].unique()
sig_map = {'ns': 0, '*': 1, '**': 2, '***': 3}
cmap = plt.cm.get_cmap('RdYlGn', 4)

for ax, comp in zip(axes, comparisons):
    sub = df[df['Comparison'] == comp]
    matrix = np.zeros((len(PKEYS), len(CONFIG['snr_levels'])))
    for i, pkey in enumerate(PKEYS):
        for j, snr in enumerate(CONFIG['snr_levels']):
            row = sub[(sub['Parameter'].str.startswith(pkey)) & (sub['SNR'] == snr)]
            if not row.empty:
                matrix[i, j] = sig_map.get(row.iloc[0]['Significance'], 0)

    im = ax.imshow(matrix, cmap=cmap, vmin=-0.5, vmax=3.5, aspect='auto')
    ax.set_xticks(range(len(CONFIG['snr_levels'])))
    ax.set_xticklabels([str(s) for s in CONFIG['snr_levels']])
    ax.set_yticks(range(len(PKEYS)))
    ax.set_yticklabels([f'{n} {u}' for n, u in zip(PARAM_NAMES, PARAM_UNITS)])
    ax.set_xlabel('SNR', fontsize=10)
    ax.set_title(comp, fontsize=9, fontweight='bold')

    # Annotate with significance stars
    for i in range(len(PKEYS)):
        for j in range(len(CONFIG['snr_levels'])):
            label = [k for k, v in sig_map.items() if v == int(matrix[i, j])][0]
            ax.text(j, i, label, ha='center', va='center', fontsize=9, fontweight='bold')

cbar = plt.colorbar(im, ax=axes, ticks=[0,1,2,3], fraction=0.02)
cbar.ax.set_yticklabels(['ns', '*', '**', '***'])
cbar.set_label('Significance (Bonferroni)', fontsize=9)

fig.suptitle('Wilcoxon signed-rank test significance\n(Bonferroni-corrected, \u03b1=0.05)',
             fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_dir'], 'significance_heatmap.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Saved significance heatmap')

In [ ]:
# ── Cell 12: Effect size (Cohen's d) summary ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

for ax, comp in zip(axes, comparisons):
    sub = df[df['Comparison'] == comp]
    d_matrix = np.zeros((len(PKEYS), len(CONFIG['snr_levels'])))
    for i, pkey in enumerate(PKEYS):
        for j, snr in enumerate(CONFIG['snr_levels']):
            row = sub[(sub['Parameter'].str.startswith(pkey)) & (sub['SNR'] == snr)]
            if not row.empty:
                d_matrix[i, j] = row.iloc[0]["Cohen's d"]

    im = ax.imshow(d_matrix, cmap='RdBu_r', aspect='auto',
                   norm=mcolors.TwoSlopeNorm(vmin=-1.5, vcenter=0, vmax=1.5))
    ax.set_xticks(range(len(CONFIG['snr_levels'])))
    ax.set_xticklabels([str(s) for s in CONFIG['snr_levels']])
    ax.set_yticks(range(len(PKEYS)))
    ax.set_yticklabels([f'{n} {u}' for n, u in zip(PARAM_NAMES, PARAM_UNITS)])
    ax.set_xlabel('SNR', fontsize=10)
    ax.set_title(comp, fontsize=9, fontweight='bold')
    for i in range(len(PKEYS)):
        for j in range(len(CONFIG['snr_levels'])):
            ax.text(j, i, f'{d_matrix[i,j]:.2f}', ha='center', va='center', fontsize=8)

cbar = plt.colorbar(im, ax=axes, fraction=0.02)
cbar.set_label("Cohen's d (negative = DL_noisy better)", fontsize=9)
fig.suptitle("Effect size (Cohen's d)", fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_dir'], 'effect_size_heatmap.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Saved effect size heatmap')